# Metrics lab

Metrics turn a route into numbers you can compare. The network finds possible paths; metrics help you decide which path is cheap, fast, low-loss, or likely to work.


In [ ]:
from dataclasses import dataclass

from simyuj.components import PortKind
from simyuj.metrics import (
    best_route,
    edge_metric,
    edge_success_probability,
    hop_count,
    link_metric,
    link_success_probability,
    route_score,
    route_success_probability,
    total_link_cost,
    total_link_delay,
    total_link_metric,
)
from simyuj.network import Network, Node
from simyuj.network.planning import candidate_routes, rank_routes
from simyuj.network.routing import RoutePlanner
from simyuj.network.topology import NetworkTopology


## 1. Start with competing paths

Start with a small graph that has a direct path and two relay paths.


In [ ]:
network = Network("metrics_lab")

for node_id in ("alice", "north", "south", "bob"):
    network.add_node(Node(node_id))

print("Nodes:", tuple(network.nodes))


In [ ]:
@dataclass(frozen=True, slots=True)
class Span:
    length_km: float
    attenuation_db_per_km: float
    connector_loss_db: float
    setup_delay_us: float

    @property
    def loss_db(self) -> float:
        return self.length_km * self.attenuation_db_per_km + self.connector_loss_db

    @property
    def propagation_delay_us(self) -> float:
        return self.length_km / 0.2

    @property
    def total_delay_us(self) -> float:
        return self.propagation_delay_us + self.setup_delay_us

    @property
    def survival(self) -> float:
        return 10 ** (-self.loss_db / 10)


In [ ]:
spans = {
    "q_alice_bob_direct": ("alice", "bob", Span(90.0, 0.20, 1.2, 4.0)),
    "q_alice_north": ("alice", "north", Span(41.0, 0.20, 0.7, 2.5)),
    "q_north_bob": ("north", "bob", Span(42.0, 0.20, 0.8, 2.5)),
    "q_alice_south": ("alice", "south", Span(34.0, 0.22, 1.3, 1.0)),
    "q_south_bob": ("south", "bob", Span(48.0, 0.22, 1.4, 1.0)),
}

for link_id, (source, target, span) in spans.items():
    network.add_quantum_link(link_id, source, target, channel=span)

print("Quantum links:")
for link_id, link in network.quantum_links.items():
    span = link.transport
    print(f"{link_id}: {link.source_node_id} -> {link.target_node_id}, {span.length_km:.1f} km")


## 2. Keep measurements by link id

Metrics use dictionaries keyed by link id. The numbers can come from physics, measurements, logs, or a sweep.


In [ ]:
length_km = {link_id: span.length_km for link_id, (_, _, span) in spans.items()}
loss_db = {link_id: span.loss_db for link_id, (_, _, span) in spans.items()}
delay_us = {link_id: span.total_delay_us for link_id, (_, _, span) in spans.items()}
survival = {link_id: span.survival for link_id, (_, _, span) in spans.items()}

print("One link, four metric values:")
link_id = "q_alice_bob_direct"
print("length_km:", length_km[link_id])
print("loss_db:", round(loss_db[link_id], 3))
print("delay_us:", round(delay_us[link_id], 3))
print("survival:", round(survival[link_id], 5))


In [ ]:
print("Reading one metric by link id:")
print("direct link length:", link_metric("q_alice_bob_direct", length_km, field_name="length_km"), "km")
print("missing maintenance penalty with default:", link_metric("q_alice_bob_direct", {}, field_name="maintenance", default=0.0))


In [ ]:
first_edge = network.outgoing_edges("alice", port_kind=PortKind.QUANTUM)[0]

print("First edge:", first_edge)
print("edge length:", edge_metric(first_edge, length_km, field_name="length_km"), "km")
print("edge success:", round(edge_success_probability(first_edge, survival), 5))


## 3. Score route candidates

Generate routes from the network. Metrics will only score these candidates.


In [ ]:
planner = RoutePlanner(NetworkTopology(network))

routes = candidate_routes(
    planner,
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    max_hops=2,
)

print("Candidate routes:")
for route in routes:
    print(route.link_ids, "nodes:", " -> ".join(route.node_ids), "hops:", hop_count(route))


In [ ]:
print("Route metric table:")
for route in routes:
    print(
        f"{route.link_ids}: "
        f"hops={hop_count(route)}, "
        f"length={total_link_metric(route, length_km, field_name='length_km'):5.1f} km, "
        f"loss={total_link_cost(route, loss_db):5.2f} dB, "
        f"delay={total_link_delay(route, delay_us):6.1f} us, "
        f"success={route_success_probability(route, survival) * 100:6.3f}%"
    )


## 4. Additive scores sum along a route

Additive metrics sum across route links.


In [ ]:
relay_route = routes[1]

print("Relay route:", relay_route.link_ids)
print("length sum:", total_link_metric(relay_route, length_km, field_name="length_km"), "km")
print("loss sum:", total_link_cost(relay_route, loss_db), "dB")
print("delay sum:", total_link_delay(relay_route, delay_us), "us")


In [ ]:
print("Success probabilities multiply:")
for route in routes:
    per_link = [round(link_success_probability(link_id, survival), 5) for link_id in route.link_ids]
    route_success = route_success_probability(route, survival)
    print(route.link_ids, "per-link", per_link, "route", round(route_success, 5))


## 5. The best route depends on the question

The best route depends on what you ask for.


In [ ]:
by_hops = best_route(routes, hop_count)
by_length = best_route(routes, lambda route: total_link_metric(route, length_km, field_name="length_km"))
by_loss = best_route(routes, lambda route: total_link_cost(route, loss_db))
by_delay = best_route(routes, lambda route: total_link_delay(route, delay_us))
by_success = best_route(routes, lambda route: 1.0 - route_success_probability(route, survival))

print("Fewest hops:", by_hops.link_ids)
print("Shortest length:", by_length.link_ids)
print("Lowest loss:", by_loss.link_ids)
print("Lowest delay:", by_delay.link_ids)
print("Highest success:", by_success.link_ids)


In [ ]:
print("Ranking by length:")
for item in rank_routes(routes, lambda route: total_link_metric(route, length_km, field_name="length_km")):
    print(f"{item.score:5.1f} km -> {item.route.link_ids}")


In [ ]:
print("Ranking by loss:")
for item in rank_routes(routes, lambda route: total_link_cost(route, loss_db)):
    print(f"{item.score:5.2f} dB -> {item.route.link_ids}")


In [ ]:
print("Ranking by success:")
ranked = rank_routes(routes, lambda route: 1.0 - route_success_probability(route, survival))
for item in ranked:
    success_percent = (1.0 - item.score) * 100
    print(f"{success_percent:6.3f}% -> {item.route.link_ids}")


## 6. Score one edge at a time

Use `route_score` when the score is easier to define edge by edge.


In [ ]:
maintenance_penalty = {
    "q_alice_north": 0.8,
    "q_north_bob": 0.8,
}

def edge_penalty(edge):
    link_loss = edge_metric(edge, loss_db, field_name="loss_db")
    maintenance = edge_metric(edge, maintenance_penalty, field_name="maintenance", default=0.0)
    return link_loss + maintenance

print("Loss plus maintenance penalty:")
for route in routes:
    print(f"{route_score(route, edge_penalty):5.2f} -> {route.link_ids}")


In [ ]:
by_penalty = best_route(routes, lambda route: route_score(route, edge_penalty))

print("Best by custom penalty:", by_penalty.link_ids)
print("Its nodes:", " -> ".join(by_penalty.node_ids))


## 7. Change the fiber and compare again

Try one more real experiment: improve the direct fiber and watch the metric choice move.


In [ ]:
improved = Span(74.0, 0.18, 0.5, 4.0)
network.add_quantum_link("q_alice_bob_new", "alice", "bob", channel=improved)

spans["q_alice_bob_new"] = ("alice", "bob", improved)
length_km["q_alice_bob_new"] = improved.length_km
loss_db["q_alice_bob_new"] = improved.loss_db
delay_us["q_alice_bob_new"] = improved.total_delay_us
survival["q_alice_bob_new"] = improved.survival

print("Added link q_alice_bob_new")
print("length:", improved.length_km, "km")
print("loss:", round(improved.loss_db, 3), "dB")
print("delay:", round(improved.total_delay_us, 3), "us")
print("success:", round(improved.survival * 100, 3), "%")


In [ ]:
updated_routes = candidate_routes(
    planner,
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    max_hops=2,
)

print("Updated route table:")
for route in updated_routes:
    print(
        f"{route.link_ids}: "
        f"length={total_link_metric(route, length_km, field_name='length_km'):5.1f} km, "
        f"loss={total_link_cost(route, loss_db):5.2f} dB, "
        f"success={route_success_probability(route, survival) * 100:6.3f}%"
    )


In [ ]:
print("New best choices:")
print("fewest hops:", best_route(updated_routes, hop_count).link_ids)
print("lowest loss:", best_route(updated_routes, lambda route: total_link_cost(route, loss_db)).link_ids)
print("highest success:", best_route(updated_routes, lambda route: 1.0 - route_success_probability(route, survival)).link_ids)


## Keep this model in your head

The habit to keep: metrics score routes. They do not search the graph, reserve resources, or change simulator state. That keeps route policy easy to swap.
